In [ ]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Input directories
# -------------------------------------------------------
ARI_dir   = r"C:\Data_for_Code\ARI_mARI_07-25-07-30-B5.tif"
Slope_dir = r"C:\Data_for_Code\slope_10m.tif"          # <-- update filename as needed
NLCD_dir  = r"C:\Data_for_Code\Annual_NLCD_LndCov_2023_CU_C1V1_d6d481be-09e8-43df-8355-b0802c92b892.tiff"

# NLCD class to restrict analysis to
DECIDUOUS = 41

# -------------------------------------------------------
# Slope class definitions
# Slope is in degrees (0-90); standard geomorphic breakpoints
# -------------------------------------------------------
SLOPE_CLASSES = [
    "Flat",          #  0 -  2 deg
    "Gentle",        #  2 -  5 deg
    "Moderate",      #  5 - 10 deg
    "Strong",        # 10 - 15 deg
    "Very Strong",   # 15 - 20 deg
    "Steep",         # 20 - 30 deg
    "Very Steep",    # 30 - 45 deg
    "Cliff",         # 45 - 90 deg
]

# Colour palette (one per class, in SLOPE_CLASSES order)
CLASS_COLORS = [
    "#a6a6a6",  # Flat         - grey
    "#abd9e9",  # Gentle       - pale blue
    "#74add1",  # Moderate     - light blue
    "#4dac26",  # Strong       - green
    "#b8e186",  # Very Strong  - light green
    "#fdae61",  # Steep        - orange
    "#f46d43",  # Very Steep   - orange-red
    "#d73027",  # Cliff        - red
]

In [ ]:
# -------------------------------------------------------
# Read ARI raster (10 m) - bands 1=ARI, 2=mARI, 3=Redness
# -------------------------------------------------------
with rasterio.open(ARI_dir) as src:
    ari_profile   = src.profile
    ari_crs       = src.crs
    ari_transform = src.transform
    ari_shape     = (src.height, src.width)
    ari_nodata    = src.nodata

    b1 = src.read(1).astype(np.float32)   # ARI
    b2 = src.read(2).astype(np.float32)   # mARI
    b3 = src.read(3).astype(np.float32)   # Redness

# Valid pixels mask for the three index bands
valid_idx = np.isfinite(b1) & np.isfinite(b2) & np.isfinite(b3)
if ari_nodata is not None:
    valid_idx &= (b1 != ari_nodata) & (b2 != ari_nodata) & (b3 != ari_nodata)

print(f"ARI grid             : {ari_shape[0]} rows x {ari_shape[1]} cols")
print(f"Valid ARI pixels     : {valid_idx.sum():,}")

In [ ]:
# -------------------------------------------------------
# Reproject NLCD (30 m) -> ARI grid (10 m), nearest neighbour
# Then restrict valid_idx to deciduous forest pixels only
# -------------------------------------------------------
nlcd_on_ari = np.full(ari_shape, fill_value=0, dtype=np.uint8)

with rasterio.open(NLCD_dir) as nl:
    nl_arr    = nl.read(1)
    nl_nodata = nl.nodata

    reproject(
        source        = nl_arr,
        destination   = nlcd_on_ari,
        src_transform = nl.transform,
        src_crs       = nl.crs,
        dst_transform = ari_transform,
        dst_crs       = ari_crs,
        resampling    = Resampling.nearest,
        src_nodata    = nl_nodata,
        dst_nodata    = 0,
    )

# Restrict the valid mask to deciduous forest only (NLCD class 41)
is_deciduous = (nlcd_on_ari == DECIDUOUS)
valid_idx    = valid_idx & is_deciduous

print(f"Deciduous pixels (NLCD 41)           : {is_deciduous.sum():,}")
print(f"Deciduous pixels with valid ARI data : {valid_idx.sum():,}")

In [ ]:
# -------------------------------------------------------
# Read & reproject Slope raster onto ARI grid
# Slope may already be 10 m; reproject ensures alignment.
# -------------------------------------------------------
slope_on_ari = np.full(ari_shape, fill_value=np.nan, dtype=np.float32)

with rasterio.open(Slope_dir) as slp:
    slp_arr    = slp.read(1).astype(np.float32)
    slp_nodata = slp.nodata

    # Replace nodata with NaN before reprojection
    if slp_nodata is not None:
        slp_arr[slp_arr == slp_nodata] = np.nan

    reproject(
        source        = slp_arr,
        destination   = slope_on_ari,
        src_transform = slp.transform,
        src_crs       = slp.crs,
        dst_transform = ari_transform,
        dst_crs       = ari_crs,
        resampling    = Resampling.bilinear,
        src_nodata    = np.nan,
        dst_nodata    = np.nan,
    )

print(f"Slope range after reproject: {np.nanmin(slope_on_ari):.1f} - {np.nanmax(slope_on_ari):.1f} degrees")

In [ ]:
# -------------------------------------------------------
# Classify slope into named classes
# Values in degrees (0-90); standard geomorphic breakpoints
# -------------------------------------------------------
def classify_slope(slp_deg):
    out   = np.full(slp_deg.shape, "", dtype=object)
    valid = np.isfinite(slp_deg)

    out[valid & (slp_deg >=  0.0) & (slp_deg <   2.0)] = "Flat"
    out[valid & (slp_deg >=  2.0) & (slp_deg <   5.0)] = "Gentle"
    out[valid & (slp_deg >=  5.0) & (slp_deg <  10.0)] = "Moderate"
    out[valid & (slp_deg >= 10.0) & (slp_deg <  15.0)] = "Strong"
    out[valid & (slp_deg >= 15.0) & (slp_deg <  20.0)] = "Very Strong"
    out[valid & (slp_deg >= 20.0) & (slp_deg <  30.0)] = "Steep"
    out[valid & (slp_deg >= 30.0) & (slp_deg <  45.0)] = "Very Steep"
    out[valid & (slp_deg >= 45.0) & (slp_deg <= 90.0)] = "Cliff"
    return out

slope_class = classify_slope(slope_on_ari)

# Report pixel counts per class (within deciduous mask only)
print("Slope class pixel counts (deciduous NLCD 41 only):")
for cls in SLOPE_CLASSES:
    n = (valid_idx & (slope_class == cls)).sum()
    print(f"  {cls:>12s} : {n:>10,} pixels")

In [ ]:
# -------------------------------------------------------
# Build DataFrame - one row per valid deciduous pixel
# -------------------------------------------------------
rows = []
for cls in SLOPE_CLASSES:
    mask = valid_idx & (slope_class == cls)
    n    = mask.sum()
    if n == 0:
        continue
    rows.append(pd.DataFrame({
        "slope_class" : cls,
        "ARI"         : b1[mask],
        "mARI"        : b2[mask],
        "Redness"     : b3[mask],
    }))

df = pd.concat(rows, ignore_index=True)

# Enforce the display order as a Categorical
df["slope_class"] = pd.Categorical(df["slope_class"],
                                    categories=SLOPE_CLASSES,
                                    ordered=True)
df = df.sort_values("slope_class").reset_index(drop=True)

print(f"Total deciduous pixels in DataFrame : {len(df):,}")
print(df.groupby("slope_class", observed=True).size().to_string())

In [ ]:
# -------------------------------------------------------
# Summary statistics per slope class
# -------------------------------------------------------
summary = df.groupby("slope_class", observed=True)[["ARI", "mARI", "Redness"]].describe()
print(summary.round(4).to_string())

In [ ]:
# -------------------------------------------------------
# FIGURE 1 - Overlapping density histograms
# One panel per index; each slope class = one histogram
# -------------------------------------------------------
plt.rcParams['font.family'] = 'Arial'

color_map = dict(zip(SLOPE_CLASSES, CLASS_COLORS))
present_classes = [c for c in SLOPE_CLASSES
                   if c in df["slope_class"].cat.categories
                   and (df["slope_class"] == c).sum() > 0]

fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

for j, col in enumerate(["ARI", "mARI", "Redness"]):
    ax = axes1[j]
    for cls in present_classes:
        vals = df.loc[df["slope_class"] == cls, col].values
        ax.hist(vals, bins=150, alpha=0.45, density=True,
                color=color_map[cls], label=cls)
    ax.set_title(f"{col} distribution by Slope", fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    if j == 0:
        ax.legend(fontsize=7, ncol=2)

fig1.suptitle("ARI / mARI / Redness - distribution by Slope Class (Deciduous forest only)",
              fontsize=13, fontweight='bold')
plt.savefig(r"C:\Data_for_Code\ARI_slope_deciduous_histograms.png", dpi=200, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

In [ ]:
# -------------------------------------------------------
# FIGURE 2 - Boxplots (slope class on X, index value on Y)
# -------------------------------------------------------
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

for j, col in enumerate(["ARI", "mARI", "Redness"]):
    ax = axes2[j]
    data_by_class = [df.loc[df["slope_class"] == cls, col].values
                     for cls in present_classes]

    bp = ax.boxplot(
        data_by_class,
        labels       = present_classes,
        showfliers   = False,
        widths       = 0.55,
        patch_artist = True,
    )
    for patch, cls in zip(bp['boxes'], present_classes):
        patch.set_facecolor(color_map[cls])
        patch.set_alpha(0.75)

    ax.set_title(f"{col} by Slope class", fontsize=11)
    ax.set_xlabel("Slope class")
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=45)

fig2.suptitle("ARI / mARI / Redness - boxplots by Slope Class (Deciduous forest only)",
              fontsize=13, fontweight='bold')
plt.savefig(r"C:\Data_for_Code\ARI_slope_deciduous_boxplots.png", dpi=200, bbox_inches='tight')
plt.show()
print("Figure 2 saved.")

In [ ]:
# -------------------------------------------------------
# FIGURE 3 - Combined 2x3 panel
# Top row: histograms | Bottom row: boxplots
# -------------------------------------------------------
fig3, axes3 = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)

for j, col in enumerate(["ARI", "mARI", "Redness"]):

    # Top row: histograms
    ax_top = axes3[0, j]
    for cls in present_classes:
        vals = df.loc[df["slope_class"] == cls, col].values
        ax_top.hist(vals, bins=150, alpha=0.45, density=True,
                    color=color_map[cls], label=cls)
    ax_top.set_title(f"{col} distribution", fontsize=11)
    ax_top.set_xlabel(col)
    ax_top.set_ylabel("Density")
    if j == 0:
        ax_top.legend(fontsize=7, ncol=2, title="Slope")

    # Bottom row: boxplots
    ax_bot = axes3[1, j]
    data_by_class = [df.loc[df["slope_class"] == cls, col].values
                     for cls in present_classes]
    bp = ax_bot.boxplot(
        data_by_class,
        labels       = present_classes,
        showfliers   = False,
        widths       = 0.55,
        patch_artist = True,
    )
    for patch, cls in zip(bp['boxes'], present_classes):
        patch.set_facecolor(color_map[cls])
        patch.set_alpha(0.75)
    ax_bot.set_title(f"{col} boxplot", fontsize=11)
    ax_bot.set_xlabel("Slope class")
    ax_bot.set_ylabel(col)
    ax_bot.tick_params(axis='x', rotation=45)

fig3.suptitle("ARI / mARI / Redness by Slope Class (Deciduous forest only)",
              fontsize=14, fontweight='bold')
plt.savefig(r"C:\Data_for_Code\ARI_slope_deciduous_combined.png", dpi=200, bbox_inches='tight')
plt.show()
print("Figure 3 (combined) saved.")

In [ ]:
# -------------------------------------------------------
# Summary statistics table
# N, Mean, Median, Std, Q25, Q75 per slope class per index
# -------------------------------------------------------
records = []
for cls in present_classes:
    sub = df[df["slope_class"] == cls]
    for col in ["ARI", "mARI", "Redness"]:
        vals = sub[col].dropna().values
        records.append({
            "Slope"  : cls,
            "Index"  : col,
            "N"      : len(vals),
            "Mean"   : np.mean(vals),
            "Median" : np.median(vals),
            "Std"    : np.std(vals, ddof=1),
            "Q25"    : np.percentile(vals, 25),
            "Q75"    : np.percentile(vals, 75),
        })

stats_df = pd.DataFrame(records)
print(stats_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))